**1. Input & Parameter Awal**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== INPUT NIM =====
nim_3digit = 125  # ganti sesuai NIM

# ===== HITUNG Knim =====
Knim =

print("Knim =", Knim)

# ===== DOMAIN =====
x = np.linspace(0, 15, 1000)

**2. Membership Function**

In [ ]:
def trimf(x, a, b, c):
    # Left shoulder
    if a == b:
        return np.where(x <= b, 1, np.maximum((c - x) / (c - b + 1e-6), 0))

    # Right shoulder
    elif b == c:
        return np.where(x >= b,  1, np.maximum((x - a) / (b - a + 1e-6), 0))

    # Normal triangle
    else:
        return np.maximum(np.minimum((x - a) / (b - a + 1e-6),(c - x) / (c - b + 1e-6)), 0)

***Desain A***

In [ ]:

a2, b2, c2 = 3, 6, 9
a3, b3, c3 = 8, 15, 15

ringan_A = trimf(x, a1, b1, c1)
sedang_A = trimf(x, a2, b2, c2)
berat_A  = trimf(x, a3, b3, c3)


***Desain B***

In [ ]:
a4, b4, c4 = 0, 0, 5 + Knim
a5, b5, c5 = 3 + Knim, 6 + Knim, 9 + Knim
a6, b6, c6 = 8 + Knim, 15 + Knim, 15 + Knim

ringan_B = trimf(x, a4, b4, c4)
sedang_B = trimf(x, a5, b5, c5)
berat_B  = trimf(x)

**3. Visualisasi Membership Function**

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(x, ringan_A, label='Ringan A')
plt.plot(x, sedang_A, label='Sedang A')
plt.plot(x, berat_A, label='Berat A')

plt.plot(x, ringan_B, '--', label='Ringan B')
plt.plot(x, sedang_B, '--', label='Sedang B')
plt.plot(x, berat_B, '--', label='Berat B')

plt.title("Perbandingan Membership Function")
plt.xlabel("Berat (kg)")
plt.ylabel("Derajat Keanggotaan")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Changed this line to move legend outside
plt.grid()
plt.xticks(np.arange(min(x), max(x)+1, 1))
plt.show()

**4. Fungsi Fuzzifikasi**

In [ ]:
def fuzzifikasi(x_val, a, b, c):
    # LEFT SHOULDER
    if a == b:
        if x_val <= b:
            return 1

        elif x_val >= c:
            return 0

        else:
            return (c - x_val) / (c - b)

    # RIGHT SHOULDER
    elif b == c:
        if x_val >= b:
            return 1

        elif x_val <= a:
            return 0

        else:
            return (x_val - a) / (b - a)

    # SEGITIGA NORMAL
    else:
        return max(min((x_val - a) / (b - a),(c - x_val) / (c - b)), 0)

**5. Rule Base + Inferensi + Defuzzifikasi**

In [ ]:
def fuzzy_system(x_val, desain='A'):

    if desain == 'A':
        r = fuzzifikasi(x_val, a1, b1, c1)
        s = fuzzifikasi(x_val, a2, b2, c2)
        b = fuzzifikasi(x_val)
    else:
        r = fuzzifikasi(x_val, a4, b4, c4)
        s = fuzzifikasi(x_val)
        b = fuzzifikasi(x_val, a6, b6, c6)

    # Output singleton (crisp mapping)
    z_ringan = 25
    z_sedang = 50
    z_berat  = 75

    # Defuzzifikasi (weighted average)
    numerator = (r*z_ringan + s*z_sedang + b*z_berat)
    denominator = (r + s + b + 1e-6) # angka 1e-6   dipakai untuk mencegah pembagian dengan nol

    output = numerator / denominator

    # Hitung membership output
    # Nilai 25, 50, 75 merupakan singleton output atau titik representasi output
    # Kategori ringan : titik representatifnya di 25
    # Kategori sedang : titik representatifnya di 50
    # Kategori berat  : titik representatifnya di 75
    mu_ringan = fuzzifikasi(output, 0, 25, 50)
    mu_sedang = fuzzifikasi(output, 25, 50, 75)
    mu_berat  = fuzzifikasi(output, 50, 75, 100)

# Cari kategori terbesar
    membership = {
      "Ringan": mu_ringan,
      "Sedang": mu_sedang,
      "Berat": mu_berat
    }

    kategori = max(membership, key=membership.get)

    return output, kategori

**6. Skenario Eksperimen**

In [ ]:
berat_uji = [1, 3, 5]

hasil = []

for b in berat_uji:
    outA, katA = fuzzy_system(b, 'A')
    outB, katB = fuzzy_system(b, 'B')

    hasil.append([b, outA, katA, outB, katB])

for row in hasil:
    print(row)

**7. Tabel Hasil (DataFrame)**

In [ ]:
import pandas as pd

df = pd.DataFrame(hasil, columns=[
    "Berat", "Output A", "Kategori A", "Output B", "Kategori B"
])

df

**8. Grafik Perbandingan Output**

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(df["Berat"], df["Output A"], marker='o', label='Desain A')
plt.plot(df["Berat"], df["Output B"], marker='s', label='Desain B')

plt.xlabel("Berat (kg)")
plt.ylabel("Output Crisp")
plt.title("Perbandingan Output Fuzzy")
plt.legend()
plt.grid()

plt.show()

**9. Selisih Output (Analisis)**

In [ ]:
df["Selisih"] = df["Output B"] - df["Output A"]
df